# Diabetes Prediction: Balanced (50/50) vs. Imbalanced Comparison

**Goal:** Train Logistic Regression, Random Forest, and XGBoost on two versions of the BRFSS 2015 dataset and compare how the class balance affects performance — especially **recall**, which is what we care about in healthcare.

**Two datasets:**
- `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` — 70,692 rows, artificially balanced 50/50
- `diabetes_binary_health_indicators_BRFSS2015.csv` — 253,680 rows, real-world imbalance (~86% no diabetes)

Both have the same 21 features and a binary target `Diabetes_binary` (0 = no diabetes, 1 = prediabetes/diabetes).

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

from xgboost import XGBClassifier

RANDOM_STATE = 42

## Load both datasets

Update the paths below to wherever your CSVs live.

In [3]:
PATH_5050 = 'diabetes_binary_5050split_health_indicators_BRFSS2015.csv'
PATH_IMBALANCED = 'diabetes_binary_health_indicators_BRFSS2015.csv'

df_bal = pd.read_csv(PATH_5050)
df_imb = pd.read_csv(PATH_IMBALANCED)

print('Balanced :', df_bal.shape, '| target split:')
print(df_bal['Diabetes_binary'].value_counts(normalize=True).round(3).to_dict())
print('\nImbalanced:', df_imb.shape, '| target split:')
print(df_imb['Diabetes_binary'].value_counts(normalize=True).round(3).to_dict())

Balanced : (70692, 22) | target split:
{0.0: 0.5, 1.0: 0.5}

Imbalanced: (253680, 22) | target split:
{0.0: 0.861, 1.0: 0.139}


## Helper: prepare features/target and split

We use `stratify=y` so the train/test split preserves the class ratio. This matters a lot on the imbalanced set — without it, a random split could give the test set a different diabetes rate than the training set and make results unstable.

In [ ]:
def make_splits(df, test_size=0.2):
    X = df.drop(columns=['Diabetes_binary']) #features - health indicators
    y = df['Diabetes_binary'] #target - 0 or 1
    return train_test_split(X, y, test_size=test_size,
                            stratify=y, random_state=RANDOM_STATE)

X_train_bal, X_test_bal, y_train_bal, y_test_bal = make_splits(df_bal)
X_train_imb, X_test_imb, y_train_imb, y_test_imb = make_splits(df_imb)

## Define models + hyperparameter grids

Each model is wrapped in a `Pipeline` with a scaler. Scaling is essential for Logistic Regression (coefficients depend on feature scale) and harmless for the tree models.

**`class_weight='balanced'`** on the imbalanced-data models tells the algorithm to penalize mistakes on the rare (diabetic) class more heavily — this is the in-model way to fight imbalance, as an alternative to resampling. XGBoost uses `scale_pos_weight` for the same purpose.

The grids are kept small so this runs in minutes, not hours. Expand them once you've confirmed the pipeline works.

In [5]:
def build_model_grids(imbalanced: bool):
    """Return {name: (pipeline, param_grid)} for all three models.
    If imbalanced=True, turn on class weighting to compensate."""

    cw = 'balanced' if imbalanced else None
    # scale_pos_weight ~ (#negatives / #positives); ~6 for the imbalanced set, 1 for balanced
    spw = 6 if imbalanced else 1

    models = {
        'LogisticRegression': (
            Pipeline([
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=2000,
                                           class_weight=cw,
                                           random_state=RANDOM_STATE))
            ]),
            {'clf__C': [0.01, 0.1, 1.0, 10.0]}
        ),
        'RandomForest': (
            Pipeline([
                ('scaler', StandardScaler()),
                ('clf', RandomForestClassifier(class_weight=cw,
                                               random_state=RANDOM_STATE,
                                               n_jobs=-1))
            ]),
            {'clf__n_estimators': [200, 400],
             'clf__max_depth': [8, 16, None],
             'clf__min_samples_leaf': [1, 5]}
        ),
        'XGBoost': (
            Pipeline([
                ('scaler', StandardScaler()),
                ('clf', XGBClassifier(scale_pos_weight=spw,
                                      eval_metric='logloss',
                                      random_state=RANDOM_STATE,
                                      n_jobs=-1))
            ]),
            {'clf__n_estimators': [200, 400],
             'clf__max_depth': [3, 6],
             'clf__learning_rate': [0.05, 0.1]}
        ),
    }
    return models

## Grid search with Stratified K-Fold, optimizing for RECALL

`StratifiedKFold` keeps each fold's class ratio equal to the whole set. `scoring='recall'` is the critical line: grid search will pick the hyperparameters that catch the most true diabetics, not the ones with the highest raw accuracy.

The function below runs the full search for one model on one dataset, then evaluates the winning model on the held-out test set.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def run_search(name, pipe, grid, X_train, y_train, X_test, y_test, scoring='recall'):
    gs = GridSearchCV(pipe, grid, scoring=scoring, cv=cv, n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]

    return {
        'model': name,
        'best_params': gs.best_params_,
        'cv_recall': gs.best_score_,
        'accuracy': accuracy_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'estimator': best,
        'y_pred': y_pred,
        'y_test': y_test,
    }

## Run everything

This trains 3 models × 2 datasets = 6 grid searches. On the imbalanced (253k-row) set the Random Forest and XGBoost searches are the slow part — expect a few minutes. Reduce grid sizes or `n_estimators` if you need it faster.

In [ ]:
results = []

# --- Balanced 50/50 dataset ---
print('=== BALANCED (50/50) ===')
for name, (pipe, grid) in build_model_grids(imbalanced=False).items():
    print(f'  running {name}...')
    r = run_search(name, pipe, grid, X_train_bal, y_train_bal, X_test_bal, y_test_bal)
    r['dataset'] = 'Balanced 50/50'
    results.append(r)

# --- Imbalanced real-world dataset ---
print('=== IMBALANCED (real-world) ===')
for name, (pipe, grid) in build_model_grids(imbalanced=True).items():
    print(f'  running {name}...')
    r = run_search(name, pipe, grid, X_train_imb, y_train_imb, X_test_imb, y_test_imb)
    r['dataset'] = 'Imbalanced'
    results.append(r)

print('done.')

=== BALANCED (50/50) ===
  running LogisticRegression...
  running RandomForest...
  running XGBoost...
=== IMBALANCED (real-world) ===
  running LogisticRegression...
  running RandomForest...


## The comparison table

This is what goes on your Google Slide. Watch the **recall** and **accuracy** columns move in opposite directions between the two datasets — that's the whole story.

In [ ]:
summary = pd.DataFrame([{
    'Dataset': r['dataset'],
    'Model': r['model'],
    'Accuracy': round(r['accuracy'], 3),
    'Recall': round(r['recall'], 3),
    'Precision': round(r['precision'], 3),
    'F1': round(r['f1'], 3),
    'ROC-AUC': round(r['roc_auc'], 3),
} for r in results])

summary = summary.sort_values(['Dataset', 'Model']).reset_index(drop=True)
summary

In [ ]:
# Export for slides / sharing
summary.to_csv('model_comparison_results.csv', index=False)
print('Saved model_comparison_results.csv')

# Also print best hyperparameters found for each run
for r in results:
    print(f"{r['dataset']:15s} | {r['model']:20s} -> {r['best_params']}")

## Confusion matrices

A confusion matrix shows the four outcomes: true negatives, false positives, false negatives, true positives. **False negatives (bottom-left) are the diabetics you missed** — the number you most want to shrink. Compare the same model across the two datasets and you'll see the balanced version pulls false negatives down at the cost of more false positives.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, r in zip(axes.ravel(), results):
    cm = confusion_matrix(r['y_test'], r['y_pred'])
    ax.imshow(cm, cmap='Blues')
    ax.set_title(f"{r['model']}\n{r['dataset']}", fontsize=10)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['No', 'Diab']); ax.set_yticklabels(['No', 'Diab'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature importance / coefficients

For the tree models, importance = how much each feature reduced prediction error. For Logistic Regression, the **coefficients** are interpretable: a positive coefficient raises the log-odds of diabetes, and `exp(coef)` gives the **odds ratio** (e.g. an odds ratio of 1.5 means a one-unit increase in that feature multiplies the odds of diabetes by 1.5, holding others fixed). Check whether the top features make clinical sense — BMI, HighBP, GenHlth, Age usually dominate. Anything surprising near the top is worth a note in your writeup.

In [ ]:
# Logistic Regression odds ratios (from the imbalanced model)
logit_result = next(r for r in results
                    if r['model'] == 'LogisticRegression' and r['dataset'] == 'Imbalanced')
logit = logit_result['estimator'].named_steps['clf']
feat_names = X_train_imb.columns

coef_df = pd.DataFrame({
    'feature': feat_names,
    'coefficient': logit.coef_[0],
    'odds_ratio': np.exp(logit.coef_[0])
}).sort_values('odds_ratio', ascending=False)
print('Logistic Regression — odds ratios (imbalanced model):')
coef_df

In [ ]:
# Random Forest feature importance (from the imbalanced model)
rf_result = next(r for r in results
                 if r['model'] == 'RandomForest' and r['dataset'] == 'Imbalanced')
rf = rf_result['estimator'].named_steps['clf']

imp_df = pd.DataFrame({
    'feature': feat_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(imp_df['feature'][::-1], imp_df['importance'][::-1])
plt.title('Random Forest feature importance (imbalanced model)')
plt.xlabel('importance')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
imp_df

## What to write in your markdown cells (per your action items)

Connect each modeling decision to a consequence. Some prompts:

- **Why stratified splitting?** Preserves the real class ratio so test scores reflect reality.
- **Why `scoring='recall'` in grid search?** A false negative (missing a diabetic) is the costly error in screening, so we tune to minimize it even if accuracy drops.
- **What does the 50/50 dataset buy us, and what does it cost?** It raises recall by forcing equal attention to both classes, but the reported accuracy no longer reflects the real-world population (only ~14% are diabetic), so accuracy on the balanced set is optimistic. This is a form of **sampling bias introduced on purpose** — good to name it explicitly.
- **`class_weight='balanced'` vs. resampling (SMOTE/downsampling):** class weighting keeps all the real data and just reweights errors; downsampling throws away majority-class rows (loses information); upsampling/SMOTE invents synthetic minority rows (can blur the decision boundary). Say which you chose and why.
- **Feature importance sanity check:** if a clinically irrelevant feature ranks high, that's a red flag for leakage or an artifact worth investigating.